# 2-D: Combining Workflow and Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AluminumShark/langgraph-skills-workshop/blob/main/en/notebooks/2-D_dual_core.ipynb)

> 📍 **Matching slides**: Session 2, "Stop 3: Security and integration", slides **25-28**; optional **43-44** (class time 1:08-1:23, about 15 minutes)

## 🎯 The mini project for these two sessions: the Homework Helper

The **Homework Helper** is a small program that helps check Python homework. You give it a piece of homework, and it replies with the review result.
In the workflow version (1-A, workflow), a syntax error gets an error reply first; only code with valid syntax goes on to the AI, which writes feedback containing **"Strengths:" and "Improvements:"**.
If the format check keeps failing, the flow stops when it reaches the maximum number of generations and marks the result as not passed. "Stopped" must never be read as "passed".

**Keep three things apart:** valid syntax (Python can read it) ≠ runs successfully (no error while running) ≠ meets the assignment (matches what the assignment asks for).

We chose it because it happens to use all three kinds of things (and you can also combine them):

| What the helper needs | Example | What we use |
| --- | --- | --- |
| Steps controlled by a flow | Check syntax first; report errors; only on a pass, write feedback and check its format | A **workflow** defined in code (LangGraph, Session 1) |
| Things that need judgment | How to word the feedback, whether to actually run the code and see | Leave it to the **AI** (agent, Session 1) |
| Ways of working and rules | Should the feedback be gentle or strict, what a good variable name looks like | A **handbook** written for the AI to read (Skill, Session 2). Rules that code can check (such as variable naming) can also come with a script or go into the workflow; 2-B compares them |

👉 **This notebook covers:** Connect the helper's workflow, agent, handbooks and scripts into one helper that combines a fixed workflow with an agent, and that can resume after an interruption.

<details><summary>Open this when you need to look up a component name (no need to memorize it)</summary>

| Component | What it does, in plain words |
| --- | --- |
| `check_syntax` (check syntax) | Only checks whether the code's "grammar" is right, for example whether brackets are closed or a colon is missing. It **does not run the code** |
| `explain_error` (explain the error) | When the syntax is wrong, tells the student in one sentence which line is wrong |
| `give_feedback` (write feedback) | Once a model is connected in 1-A, asks the AI to write "Strengths + Improvements" feedback; the next station checks whether it matches the format |
| `check_format` (check format) | Checks the order of the two markers, that each part is non-empty, and the total character count. It does not verify that the feedback content is correct |
| Tools: `run_code`, `check_indent`, `read_spec` (from 1-B on); `check_naming` (from 2-B on) | Run the homework under teaching limits, check indentation, read the assignment requirements, check variable names |
| Handbooks: `friendly-feedback`, `strict-review`; `naming-check` (2-B), `detailed-feedback` (the 2-C security experiment) | Tell the AI whether feedback should be gentle and encouraging or strict; the naming-check handbook comes with a script; the last one hides a suspicious instruction so you can practice checking an outside handbook |

</details>

**Reminder:** run "0. Setup" first, then follow each part: setup cell → predict → run → compare.
`%%writefile` means "save this cell's content as a file"; edits only take effect after you run the cell. Re-running a setup cell may overwrite the handbooks, so save your own edits first.

## What this notebook does

You can build the helper in two ways:

- **Workflow**: the route is drawn in code (LangGraph). It checks syntax first; only if the syntax passes does it write feedback and check the format. Feedback that fails can be sent back to be rewritten, and the flow stops when it reaches the maximum number of generations.
  The branches and the order are clear; for any case you haven't handled, you still have to add a flow rule. (Session 1, 1-A)
- **Agent**: you give the AI only a goal and a toolbox, and it decides which tools to use and how many times. It is very flexible, but the path can differ from run to run. (Session 1, 1-B)

This notebook connects a fixed workflow with an agent. That is the **dual-core** approach from the course title:

- **Outside** is the workflow (the program controls branches and order): check syntax first; only a pass goes on to the agent; the feedback must pass the format check, and the flow stops at the generation limit, possibly still not passed.
- **Inside**, the `give_feedback` station (which writes the feedback) is a small agent (AI judgment):
  the handbook is 2-A's "friendly feedback" or "strict review", and its toolbox has 2-B's naming-check script `check_naming`, `run_code` (runs the homework), and `read_spec` (reads the assignment requirements).
  It decides on its own whether to check naming and whether to read the assignment.
- The assignment and the homework are paired when a run starts, and the assignment is saved in the State as `spec`. `read_spec` reads only the assignment in that State, not the shared `spec.md`; a resumed run still uses the assignment in the original checkpoint.
- A new component, `skill_loader`: one station in the workflow that looks at the **cards** of all handbooks (the name and description at the top of `SKILL.md`), picks one, and hands its body to the next station.
- Finally, a **checkpointer** (automatic saving) and a **thread_id** (the number of a save) store the **State** after every step (the "order slip" passed between stations, a dict):
  if a run is interrupted, it can continue.

| Part | What you'll see | Time | Matching slides |
| --- | --- | --- | --- |
| Part 1 | Write `dual_core.py` and draw the combined flow chart | 2 min | 25-26 |
| Part 2 | **Demo**: run the whole thing once; the outer flow goes station by station, and the agent inside picks its own tools | 3 min | 25-26 |
| Part 3 | **Experiment**: interrupt; stop halfway and save the State to the checkpoint | 2 min | 27 |
| Part 4 ⭐ | **Experiment**: resume with the same thread_id, or start from blank with a different thread_id | 5 min | 27-28 |
| Part 5 | **Inspect**: the State history of one thread (one save per step) | 2 min | 28 |
| Part 6 | Optional: use a separate pure agent for an eval and compare prompt versions; record the pass rate | - | 43-44 |

## How to use it

1. Run **"0. Setup"** first (this cell prepares a **fresh set of default copies**; it does not import the handbooks, scripts or tools you changed in earlier notebooks). This notebook uses the default handbooks.
2. The only thing you need to change in this notebook is the **thread_id** (the forms in Parts 3 and 4). You can change the other forms too.
   Optional: to use the handbook you changed in 2-A, finish Parts 1-4 first, then open `/content/review-bot/skills/friendly-feedback/SKILL.md` in the left "Files" panel,
   paste in the content you backed up and save it (don't re-run "0. Setup" afterward, or it will be overwritten), and then re-run from Part 2 with a brand-new thread_id.
3. The setup and prepare cells overwrite the handbooks, scripts and code in them, so keep any changes you made yourself; they do not delete an existing SQLite checkpoint.
4. Each part: **🤔 write your prediction → ▶ run → 👀 compare**; in Parts 3 and 4, look at the checkpoint state before you act; to run an experiment again, use a thread_id you have never used.
5. **Class pace**: about 15 minutes for Parts 1-4, a quick look at the history in Part 5, and Part 6 (eval) is optional after class.

In [ ]:
#@title 0. Setup (run it the first time you open this notebook, or after the runtime is reset)
#@markdown **Model source**: `colab` = the model built into Colab (no API key needed; availability and quota depend on your account); `gemini` = the instructor's Gemini API key (the backup option)
model_source = "colab"  #@param ["colab", "gemini"]
#@markdown If you pick `gemini`: we recommend adding `GOOGLE_API_KEY` under 🔑 "Secrets" in the left sidebar. You can also paste the key below (⚠️ if you paste it, do not share this notebook)
INSTRUCTOR_API_KEY = ""  #@param {type:"string"}
#@markdown Advanced (you usually don't need to change these): set a model name, or leave it empty to use the default
colab_model_name = ""  #@param {type:"string"}
gemini_model_name = "gemini-flash-latest"  #@param {type:"string"}

import os, shutil, subprocess, sys
from pathlib import Path

# 1. Install packages with uv (limited to compatible version ranges; time depends on your network)
if not os.environ.get("WORKSHOP_SKIP_INSTALL"):
    print("⏳ Installing LangGraph / LangChain with uv ...")
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(f'uv pip install --python "{sys.executable}" -q "langgraph>=1.1,<2" "langchain>=1.2,<2" "langchain-google-genai>=4,<5" "langgraph-checkpoint-sqlite>=3,<4"', shell=True, check=True)

# 2. Write the shared files this notebook uses
RB = Path("/content/review-bot")
for d in ['tools', 'deploy', 'skills/friendly-feedback', 'skills/strict-review', 'skills/naming-check/scripts', 'skills/detailed-feedback/scripts']:
    (RB / d).mkdir(parents=True, exist_ok=True)
_files = {
    'llm.py': r'''"""Model adapter: wraps either "Colab's built-in model" or "the instructor's Gemini API key" into one model object.

Students don't need to read this file. Every later cell only writes model.invoke(...) and model.bind_tools(...),
and it works the same no matter which provider is behind it.

- colab : google.colab.ai.generate_text (works with a free account, no API key needed)
- gemini: langchain_google_genai.ChatGoogleGenerativeAI (needs GOOGLE_API_KEY)

generate_text takes only one piece of text and returns only one piece of text. It has no native tool calling.
So ColabAIChat adds a small adapter:
1. It joins the messages into one text (system instructions, user, AI, and tool results each get a header)
2. When tools are bound, it appends the tool list and asks the model to "reply with a single line of JSON if you need a tool"
3. It parses the model's JSON into tool_calls, in exactly the same format as native tool calling
This lets you see that the model only ever "says" which tool it wants.
"""

from __future__ import annotations

import json
import os
import re
import time
import uuid
from pathlib import Path
from typing import Any

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool

SETTINGS_FILE = Path(__file__).with_name(".llm.json")
DEFAULT_GEMINI_MODEL = "gemini-flash-latest"


def estimate_tokens(text: str) -> int:
    """Rough token estimate: about 4 characters per token for English text.
    This is only a guess. A real tokenizer can count noticeably more or fewer."""
    return max(1, round(len(text) / 4))


# ---------- Join the messages into one text ----------

def _describe_tools(tools: list[dict]) -> str:
    lines = []
    for t in tools:
        fn = t.get("function", t)
        props = fn.get("parameters", {}).get("properties", {})
        params = ", ".join(f"{k}: {v.get('type', 'string')}" for k, v in props.items())
        desc = " ".join((fn.get("description") or "").split())
        lines.append(f"- {fn['name']}({params}): {desc}")
    return "\n".join(lines)


def _messages_to_text(messages: list[BaseMessage], tools: list[dict] | None, tool_choice: Any) -> str:
    parts = []
    for m in messages:
        if m.type == "system":
            parts.append(f"=== System instructions ===\n{m.text}")
        elif m.type == "human":
            parts.append(f"=== User ===\n{m.text}")
        elif m.type == "ai":
            if m.tool_calls:
                calls = [{"name": c["name"], "args": c["args"]} for c in m.tool_calls]
                parts.append("=== AI ===\n" + json.dumps({"tool_calls": calls}, ensure_ascii=False))
            else:
                parts.append(f"=== AI ===\n{m.text}")
        elif m.type == "tool":
            name = getattr(m, "name", None) or "tool"
            parts.append(f"=== Result of tool {name} ===\n{m.text}")
    if tools:
        must = ""
        if tool_choice in ("any", "required"):
            must = "This time you must call at least one tool.\n"
        elif isinstance(tool_choice, str) and tool_choice not in ("auto", "none"):
            must = f"This time you must call {tool_choice}.\n"
        parts.append(
            "=== Tools you can use ===\n"
            f"{_describe_tools(tools)}\n\n"
            "=== Reply rules ===\n"
            f"{must}"
            "If you need to use a tool, reply with a single line of JSON only. Add no other text and no ``` marks:\n"
            '{"tool_calls": [{"name": "tool name", "args": {"parameter name": "parameter value"}}]}\n'
            "If you don't need a tool (for example, you already have enough information), just answer in plain text and output no JSON."
        )
    parts.append("=== AI ===")
    return "\n\n".join(parts)


def _parse_reply(text: str, tools: list[dict] | None) -> AIMessage:
    """If the model replied with a JSON note, turn it into tool_calls. Otherwise treat it as plain text."""
    meta = {"raw_text": text, "model_provider": "colab"}
    if tools:
        s = text.strip()
        s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s)
        start, end = s.find("{"), s.rfind("}")
        if start != -1 and end > start and ("tool_calls" in s or s.startswith("{")):
            try:
                data = json.loads(s[start : end + 1])
            except json.JSONDecodeError:
                data = None
            calls = None
            if isinstance(data, dict):
                if isinstance(data.get("tool_calls"), list):
                    calls = data["tool_calls"]
                elif "name" in data or "tool" in data:
                    calls = [data]
            if calls:
                tool_calls = []
                for c in calls:
                    if not isinstance(c, dict):
                        continue
                    name = c.get("name") or c.get("tool")
                    args = c.get("args") or c.get("arguments") or {}
                    if isinstance(args, str):
                        try:
                            args = json.loads(args)
                        except json.JSONDecodeError:
                            args = {}
                    if name:
                        tool_calls.append(
                            {"name": name, "args": args, "id": f"call_{uuid.uuid4().hex[:8]}", "type": "tool_call"}
                        )
                if tool_calls:
                    return AIMessage(content="", tool_calls=tool_calls, response_metadata=meta)
    return AIMessage(content=text, response_metadata=meta)


class ColabAIChat(BaseChatModel):
    """Wraps google.colab.ai.generate_text as a LangChain chat model."""

    model_name: str | None = None
    max_retries: int = 3

    @property
    def _llm_type(self) -> str:
        return "colab-ai"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        formatted = [convert_to_openai_tool(t) for t in tools]
        return self.bind(tools=formatted, tool_choice=tool_choice)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        from google.colab import ai

        tools = kwargs.get("tools")
        prompt = _messages_to_text(messages, tools, kwargs.get("tool_choice"))
        extra = {"model_name": self.model_name} if self.model_name else {}
        for attempt in range(self.max_retries):
            try:
                text = ai.generate_text(prompt, **extra) or ""
                break
            except Exception:
                if attempt == self.max_retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
        message = _parse_reply(text, tools)
        message.usage_metadata = {
            "input_tokens": estimate_tokens(prompt),
            "output_tokens": estimate_tokens(text),
            "total_tokens": estimate_tokens(prompt) + estimate_tokens(text),
        }
        message.response_metadata["token_count"] = "estimate (Colab's built-in model does not report usage)"
        return ChatResult(generations=[ChatGeneration(message=message)])


# ---------- Public: settings and getting the model ----------

def load_settings() -> dict:
    try:
        return json.loads(SETTINGS_FILE.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def save_settings(provider: str, colab_model: str = "", gemini_model: str = "") -> None:
    """Remember which model provider was chosen (the API key is not saved)."""
    SETTINGS_FILE.write_text(
        json.dumps(
            {"provider": provider, "colab_model": colab_model, "gemini_model": gemini_model or DEFAULT_GEMINI_MODEL},
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def _find_api_key() -> str | None:
    if os.environ.get("GOOGLE_API_KEY"):
        return os.environ["GOOGLE_API_KEY"]
    try:
        from google.colab import userdata

        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return None


def get_model() -> BaseChatModel:
    cfg = load_settings()
    # In the cloud there is no settings file: use the environment variable WORKSHOP_PROVIDER=gemini (the key is read from GOOGLE_API_KEY)
    provider = os.environ.get("WORKSHOP_PROVIDER") or cfg.get("provider", "colab")
    if provider == "gemini":
        key = _find_api_key()
        if not key:
            raise RuntimeError(
                "You chose gemini but no API key was found. Add GOOGLE_API_KEY under 🔑 Secrets on the left, "
                "or paste the key from the instructor into the form in \"0. Environment setup\", then rerun that cell."
            )
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model=cfg.get("gemini_model") or DEFAULT_GEMINI_MODEL, google_api_key=key, max_retries=3
        )
    try:
        from google.colab import ai  # noqa: F401
    except ImportError as exc:
        raise RuntimeError("google.colab.ai only works on Google Colab. If you are not on Colab, choose gemini instead.") from exc
    return ColabAIChat(model_name=cfg.get("colab_model") or None)
''',
    'workshop.py': r'''"""Small helpers shared by the workshop notebooks (students don't need to read this).

- prepare()                     : the first cell of every part calls it to reconnect the model and reload edited files
- sample_homework               : the sample homework in the form dropdown
- record() / print_history()    : save the results that experiment cells name, then list them by title in the last cell
- show_graph()                  : draw a LangGraph as a diagram
- save_to_drive() / download_zip()
"""

from __future__ import annotations

import datetime as _dt
import importlib
import json
import shutil
import sys
from pathlib import Path

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import BaseMessage

from llm import get_model, estimate_tokens  # noqa: F401  (so notebooks can write: from workshop import estimate_tokens)

__all__ = [
    "BASE", "prepare", "sample_homework", "load_homework", "record", "print_state", "print_order_slip", "print_history", "clear_history",
    "ModelCallCounter", "show_graph", "save_to_drive", "download_zip", "estimate_tokens",
]

BASE = Path(__file__).resolve().parent
HISTORY_FILE = BASE / "history.jsonl"

sample_homework = {
    "Missing parenthesis": "print('hi'\n",
    "Correct program": "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n",
    "Empty": "",
    "Variable typo": "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n",
    "Messy indentation": "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n  return total / len(scores)\n\nprint(average([90, 85, 77]))\n",
    "Inconsistent naming": (
        "def CalcAverage(scoreList):\n"
        "    Total = 0\n"
        "    for s in scoreList:\n"
        "        Total += s\n"
        "    avgScore = Total / len(scoreList)\n"
        "    return avgScore\n"
        "\n"
        "myScores = [90, 85, 77]\n"
        "MAX = 100\n"
        "result_2 = CalcAverage(myScores)\n"
        "print(result_2, '/', MAX)\n"
    ),
}


def load_homework(name: str) -> str:
    """When the form says "My homework", read /content/review-bot/my_homework.py. Otherwise take it from the samples."""
    if name.startswith("My homework"):
        path = BASE / "my_homework.py"
        if not path.exists():
            raise FileNotFoundError('There is no "My homework" yet. Run the cell above that has %%writefile ... my_homework.py first.')
        return path.read_text(encoding="utf-8")
    return sample_homework[name]


def prepare():
    """The start of every part: put review-bot on the import path, reload edited files, and return the model."""
    if str(BASE) not in sys.path:
        sys.path.insert(0, str(BASE))
    importlib.invalidate_caches()
    for name, mod in list(sys.modules.items()):
        path = getattr(mod, "__file__", None) or ""
        if path and Path(path).resolve().is_relative_to(BASE) and name not in ("workshop", "llm"):
            try:
                importlib.reload(mod)
            except Exception:
                sys.modules.pop(name, None)
    _make_mcp_stdio_work_in_notebook()
    return get_model()


def _make_mcp_stdio_work_in_notebook() -> None:
    """mcp's stdio_client connects the MCP server's error output to sys.stderr.
    On Colab (Linux), sys.stderr has a real file descriptor, so this works. Jupyter on Windows has none, so we redirect to a log file."""
    try:
        sys.stderr.fileno()
        return
    except Exception:
        pass
    try:
        import mcp.client.stdio as mcp_stdio
    except ImportError:
        return
    (BASE / "deploy").mkdir(exist_ok=True)
    log = open(BASE / "deploy" / "mcp_server.log", "a", encoding="utf-8")
    mcp_stdio.stdio_client.__wrapped__.__defaults__ = (log,)


# ---------- State history ----------

def _serializable(obj):
    if isinstance(obj, BaseMessage):
        d = {"type": obj.type, "content": obj.text}
        if getattr(obj, "tool_calls", None):
            d["tool_calls"] = [{"name": c["name"], "args": c["args"]} for c in obj.tool_calls]
        if obj.type == "tool":
            d["name"] = getattr(obj, "name", None)
        return d
    if isinstance(obj, dict):
        return {str(k): _serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_serializable(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def record(title: str, content) -> None:
    """Save this result (usually a State) into history.jsonl."""
    row = {"time": _dt.datetime.now().strftime("%H:%M:%S"), "title": title, "content": _serializable(content)}
    with HISTORY_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def _short(text, n=300):
    text = str(text)
    return text if len(text) <= n else text[:n] + f"... ({len(text) - n} more characters)"


def print_state(state: dict, indent: str = "  ") -> None:
    for k, v in state.items():
        if k == "messages":
            print(f"{indent}{k}: ({len(v)} messages)")
            for m in v:
                m = _serializable(m)
                if isinstance(m, dict) and m.get("tool_calls"):
                    print(f"{indent}  [{m['type']}] calls {m['tool_calls']}")
                elif isinstance(m, dict):
                    print(f"{indent}  [{m.get('type')}] {_short(m.get('content', ''), 200)}")
        else:
            print(f"{indent}{k}: {_short(repr(v) if not isinstance(v, str) else repr(v), 300)}")


def print_order_slip(step_no: int, state: dict, prev_state: dict | None = None, skip_keys: tuple = ("code",)) -> None:
    """stream_mode="values" gives the whole order slip at every step. This also marks which fields this step newly wrote."""
    if not prev_state:
        note = "start: blank order slip"
    else:
        changed_keys = [k for k in state if state.get(k) != prev_state.get(k)]
        note = "newly written: " + (", ".join(changed_keys) if changed_keys else "(nothing changed)")
    print(f"Step {step_no} ({note})")
    print("  ", {k: (_short(v, 120) if isinstance(v, str) else v) for k, v in state.items() if k not in skip_keys})


def print_history() -> None:
    if not HISTORY_FILE.exists():
        print("There are no records yet. Run a few parts above first, then come back to this cell.")
        return
    rows = [json.loads(line) for line in HISTORY_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"📸 {len(rows)} records (saved in {HISTORY_FILE})\n")
    for i, row in enumerate(rows, 1):
        print(f"===== {i}. [{row['time']}] {row['title']} =====")
        content = row["content"]
        if isinstance(content, dict):
            print_state(content)
        elif isinstance(content, list):
            for j, item in enumerate(content, 1):
                print(f"  --- step {j} ---")
                if isinstance(item, dict):
                    print_state(item, "    ")
                else:
                    print(f"    {_short(item)}")
        else:
            print(f"  {_short(content, 1000)}")
        print()


def clear_history() -> None:
    HISTORY_FILE.unlink(missing_ok=True)


# ---------- Count how many times the model was called ----------

class ModelCallCounter(BaseCallbackHandler):
    """Put it in config={"callbacks": [counter]}. It adds 1 every time the model is called."""

    def __init__(self):
        self.count = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.count += 1

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.count += 1


# ---------- Draw the graph ----------

def show_graph(graph) -> None:
    g = graph.get_graph()
    try:
        from IPython.display import Image, display

        display(Image(g.draw_mermaid_png()))
    except Exception:
        print("(The drawing service is unreachable for now, so here is a text version. You can paste it into https://mermaid.live to see the diagram.)\n")
        print(g.draw_mermaid())


# ---------- Saving ----------

def save_to_drive() -> None:
    from google.colab import drive

    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/langgraph-skills-workshop") / _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    shutil.copytree(BASE, destination, ignore=shutil.ignore_patterns("__pycache__", ".llm.json"))
    print(f"✅ Saved to Google Drive: {destination}")
    print('   (Save the notebook itself with the menu above: "File → Save a copy in Drive")')


def download_zip() -> None:
    from google.colab import files

    name = BASE.parent / ("review-bot-" + _dt.datetime.now().strftime("%Y%m%d-%H%M%S"))
    zip_path = shutil.make_archive(str(name), "zip", BASE.parent, BASE.name)
    print(f"✅ Packed: {zip_path}")
    files.download(zip_path)
''',
    'tools/__init__.py': r'''"""The Homework Helper's toolbox: run_code, check_indent, read_spec, check_naming."""
''',
    'tools/run_code.py': r'''"""run_code: runs a student's homework under teaching restrictions.

A student's homework could contain anything (import os to delete files, an infinite loop), so this tool has two execution limits plus one readability feature:
1. A check before running: only a few standard modules may be imported (math, random, re, ...). os / sys / subprocess and the like are always blocked.
   eval, exec, open, input and attributes that start with an underscore (often used to get around restrictions) are not allowed either
2. The code runs in a separate child process, and it is stopped by force when the time runs out (so an infinite loop can be stopped too)
3. It returns only one line of result (which line, what error), which is easier for people and models to read. This is not a security limit

⚠️ This is a teaching fence. For a real product, use a real sandbox (a container, RestrictedPython, e2b, Modal).
"""

import ast
import json
import os
import subprocess
import sys
import tempfile

from langchain.tools import tool

BLOCKED_MODULES = {"os", "sys", "subprocess", "shutil", "socket", "pathlib", "ctypes", "importlib",
           "multiprocessing", "threading", "signal", "pickle", "builtins", "io", "inspect", "gc"}
ALLOWED_MODULES = {"math", "random", "string", "re", "datetime", "collections", "itertools", "functools",
           "statistics", "time", "decimal", "fractions", "json", "copy", "heapq", "bisect"}
BLOCKED_FUNCTIONS = {"eval", "exec", "compile", "open", "input", "__import__", "globals", "locals", "vars",
           "getattr", "setattr", "delattr", "breakpoint", "help", "exit", "quit", "memoryview"}
BLOCKED_ATTRIBUTES = {"system", "popen", "modules", "builtins", "f_globals", "f_back", "gi_frame", "cr_frame"}
TIME_LIMIT = 3
MAX_OUTPUT_CHARS = 800

# The code in the child process that really runs the homework: it uses a "shrunken builtins", and import lets only the allowlist through
_RUNNER = r"""
import builtins, contextlib, io, json, sys, traceback
code = sys.stdin.buffer.read().decode("utf-8")
allowed = set(json.loads(sys.argv[1]))
blocked = set(json.loads(sys.argv[2]))
real_import = builtins.__import__
def guarded_import(name, globals=None, locals=None, fromlist=(), level=0):
    if name.split(".")[0] not in allowed:
        raise ImportError("This tool does not allow import " + name)
    return real_import(name, globals, locals, fromlist, level)
safe = {k: v for k, v in vars(builtins).items() if k not in blocked}
safe["__import__"] = guarded_import
try:
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (512 * 1024 * 1024, 512 * 1024 * 1024))
except Exception:
    pass
out = io.StringIO()
result = {"ok": True}
try:
    compiled = compile(code, "homework.py", "exec")
    with contextlib.redirect_stdout(out):
        exec(compiled, {"__builtins__": safe, "__name__": "__main__"})
except SyntaxError as e:
    result = {"ok": False, "line": e.lineno, "error": "SyntaxError: " + str(e.msg)}
except BaseException as e:
    line = None
    for frame in traceback.extract_tb(e.__traceback__):
        if frame.filename == "homework.py":
            line = frame.lineno
    result = {"ok": False, "line": line, "error": type(e).__name__ + ": " + str(e)}
result["stdout"] = out.getvalue()
sys.__stdout__.write(json.dumps(result))
"""


def _precheck(code: str) -> str | None:
    """A static check before running. If there is a problem, return a one-line explanation. Otherwise return None."""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"❌ Line {e.lineno} SyntaxError: {e.msg}"
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            names = [a.name for a in node.names] if isinstance(node, ast.Import) else [node.module or ""]
            for name in names:
                root = name.split(".")[0]
                if root in BLOCKED_MODULES or root not in ALLOWED_MODULES:
                    return f"⛔ Blocked: line {node.lineno} import {name} (this tool does not allow {root})"
        elif isinstance(node, ast.Name) and node.id in BLOCKED_FUNCTIONS:
            return f"⛔ Blocked: line {node.lineno} uses {node.id} (this tool does not allow it)"
        elif isinstance(node, ast.Attribute) and (
            node.attr.startswith("_") or node.attr in BLOCKED_ATTRIBUTES or node.attr in BLOCKED_MODULES
        ):
            return f"⛔ Blocked: line {node.lineno} accesses .{node.attr} (this tool does not allow it)"
    return None


def run_homework(code: str, time_limit_seconds: float = TIME_LIMIT) -> str:
    """The function that does the real work: check → run in a child process → return one line of result."""
    if not code.strip():
        return "❌ The homework is empty, there is nothing to run"
    blocked = _precheck(code)
    if blocked:
        return blocked
    env = {"PYTHONIOENCODING": "utf-8", "PATH": os.environ.get("PATH", "")}
    if os.name == "nt":
        env["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", "")
    try:
        proc = subprocess.run(
            [sys.executable, "-I", "-c", _RUNNER, json.dumps(sorted(ALLOWED_MODULES)), json.dumps(sorted(BLOCKED_FUNCTIONS))],
            input=code, capture_output=True, text=True, encoding="utf-8",
            timeout=time_limit_seconds, cwd=tempfile.gettempdir(), env=env,
        )
    except subprocess.TimeoutExpired:
        return f"⏱️ Ran longer than {time_limit_seconds:g} seconds and was stopped by force (there may be an infinite loop)"
    try:
        result = json.loads(proc.stdout.strip().splitlines()[-1])
    except (IndexError, json.JSONDecodeError):
        return "❌ The run failed (the program did not end normally; it may have used too much memory)"
    stdout = result.get("stdout", "")
    if len(stdout) > MAX_OUTPUT_CHARS:
        stdout = stdout[:MAX_OUTPUT_CHARS] + "\n... (output too long, the rest is omitted)"
    if result["ok"]:
        return f"✅ Ran successfully, output:\n{stdout}" if stdout else "✅ Ran successfully (nothing was printed)"
    where = f"line {result['line']} " if result.get("line") else ""
    output = f"\n(printed before the error: {stdout.strip()})" if stdout.strip() else ""
    return f"❌ {where}{result['error']}{output}"


@tool
def run_code(code: str) -> str:
    """Run a piece of Python homework under teaching restrictions and return the output or the error message.

    When to use it: when you want to know what the homework actually prints when it runs, or whether it errors.
    Parameter code: the complete Python code (a string).
    Returns: one line of result, for example "✅ Ran successfully, output: 5" or "❌ line 3 NameError: ...".
    Limits: it can only import a few standard modules such as math, random and re; it cannot use input() or open(); it is stopped after 3 seconds.
    """
    return run_homework(code)
''',
    'tools/check_naming.py': r'''"""check_naming: wraps the Skill's script scripts/check_naming.py as an agent tool.

Wrap the script in @tool and it joins the agent's toolbox.
In this example the AI decides whether to call the tool and tidies up the result. With the same version and the same input, the Python script returns the same check result,
but you still need to confirm whether the model actually called it and whether it kept the result intact.
"""

import subprocess
import sys
from pathlib import Path

from langchain.tools import tool

SCRIPT = Path(__file__).resolve().parent.parent / "skills" / "naming-check" / "scripts" / "check_naming.py"


def run_naming_script(code: str) -> str:
    proc = subprocess.run(
        [sys.executable, str(SCRIPT)], input=code.encode("utf-8"), capture_output=True, timeout=10
    )
    output = proc.stdout.decode("utf-8").strip()
    return output or "The script failed to run: " + proc.stderr.decode("utf-8", "replace").strip()[-200:]


@tool
def check_naming(code: str) -> str:
    """Check whether variable and function names are lowercase with underscores (snake_case), and return the lines that break the rule.

    When to use it: when reviewing a homework and you need to list the naming problems (this is the script that comes with the naming-check handbook).
    Parameter code: the complete Python code (a string).
    Returns: one line per problem, for example "Line 3: myVar breaks the rule (suggest my_var)". If everything passes it returns "OK: ...".
    """
    return run_naming_script(code)
''',
    'graph.py': r'''"""1-A: the Homework Helper as a workflow. The route is decided by code.

START → check_syntax ─syntax OK→ give_feedback → check_format ─format OK or limit reached→ END
                     └syntax error→ explain_error → END       └failed and retries below the limit→ send back to give_feedback

Three components:
- State: one order slip (a dict) that travels from start to finish
- Node : one workstation (a function). It takes the State and returns only the fields it changed
- Edge : the arrows between workstations. A conditional edge is just an if that returns the name of the next station
"""

import ast
import re
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict, total=False):
    code: str              # the homework the student submitted
    syntax_ok: bool | None # is the syntax valid? (None before it is checked)
    error: str             # where the syntax is wrong
    feedback: str          # the feedback for the student
    format_ok: bool | None # does the feedback pass the format check?
    format_problem: str    # why it failed (told to the AI when the feedback is sent back)
    retries: int           # how many times feedback has been generated, including the first


# ---------- Node 1: check_syntax (pure Python): only checks the grammar, never runs the code ----------

_ERROR_TRANSLATIONS = [
    (r"'(.)' was never closed", lambda m: f"missing a closing bracket {dict(zip('([{', ')]}')).get(m.group(1), '')}"),
    (r"unterminated string literal", lambda m: "the string is missing its closing quote"),
    (r"expected ':'", lambda m: "missing a colon :"),
    (r"unexpected indent", lambda m: "extra indentation"),
    (r"expected an indented block", lambda m: "this line needs to be indented"),
    (r"unindent does not match", lambda m: "the indentation doesn't line up"),
    (r"unmatched '(.)'", lambda m: f"an extra {m.group(1)}"),
    (r"invalid character", lambda m: "there is a full-width or unrecognized character"),
    (r"Perhaps you forgot a comma", lambda m: "maybe a comma is missing"),
    (r"invalid syntax", lambda m: "invalid syntax"),
]


def translate_syntax_error(msg: str) -> str:
    for pattern, wording in _ERROR_TRANSLATIONS:
        m = re.search(pattern, msg)
        if m:
            return wording(m)
    return msg


def check_syntax(state: State) -> dict:
    code = state.get("code", "")
    if not code.strip():
        return {"syntax_ok": False, "error": "the homework is empty, there is no code"}
    try:
        ast.parse(code)
    except SyntaxError as e:
        return {"syntax_ok": False, "error": f"line {e.lineno}: {translate_syntax_error(e.msg)}"}
    return {"syntax_ok": True, "error": ""}


# ---------- Conditional edge 1: syntax OK → write feedback; syntax error → explain the error (just an if) ----------

def route_after_syntax(state: State) -> str:
    if state["syntax_ok"]:
        return "give_feedback"
    return "explain_error"


# ---------- Node 2: explain_error (pure Python): one sentence saying which line is wrong ----------

def explain_error(state: State) -> dict:
    return {"feedback": f"Syntax: {state['error']}\nImprovements: fix this line first, then run it again."}


# ---------- Node 3: give_feedback (two versions: fixed text / handed to the AI) ----------

def give_feedback_fixed_text(state: State) -> dict:
    """The version with no AI yet: it returns a fixed piece of text."""
    return {
        "feedback": "Strengths: this homework passes the syntax check.\nImprovements: (this version has no AI yet, so the run result and the logic are not checked)",
        "retries": state.get("retries", 0) + 1,
    }


def make_give_feedback(model, feedback_char_limit: int):
    """The version with AI: the AI is just one line of code living inside this node."""

    def give_feedback(state: State) -> dict:
        prompt = (
            "You are a Python homework helper. Give feedback in English on the homework below.\n"
            "Format rules (you must follow them):\n"
            '- Write only two paragraphs: the first starts with "Strengths:", the second starts with "Improvements:"\n'
            f'- The whole feedback (including "Strengths:", "Improvements:", and line breaks) must be at most {feedback_char_limit} characters\n\n'
            f"Homework:\n```python\n{state['code']}\n```"
        )
        if state.get("format_problem"):
            prompt += f"\n\nYour previous feedback was rejected because: {state['format_problem']}. Fix it and rewrite."
        text = model.invoke(prompt).text
        return {"feedback": text.strip(), "retries": state.get("retries", 0) + 1}

    return give_feedback


# ---------- Node 4: check_format (a pure-Python checkpoint): needs two paragraphs and must not be too long ----------

def make_check_format(feedback_char_limit: int):
    def check_format(state: State) -> dict:
        feedback = state.get("feedback", "")
        problems = []
        # Check the label order and that both paragraphs are non-empty; line breaks are accepted.
        parts = re.fullmatch(r"\s*Strengths:\s*(.+?)\s*Improvements:\s*(.+?)\s*", feedback, re.DOTALL)
        if not parts or not all(part.strip() for part in parts.groups()):
            problems.append('it needs two paragraphs in order: "Strengths: ..." and "Improvements: ..."')
        if len(feedback) > feedback_char_limit:
            problems.append(f"too long: {len(feedback)} characters (including labels and line breaks), limit {feedback_char_limit} characters")
        return {"format_ok": not problems, "format_problem": "; ".join(problems)}

    return check_format


# ---------- Conditional edge 2: format OK or generation limit reached → end; otherwise send back to rewrite ----------

def make_route_after_format(max_retries: int):
    def route_after_format(state: State) -> str:
        if state["format_ok"]:
            return END
        if state.get("retries", 0) >= max_retries:
            return END  # reached the feedback generation limit; stopping does not mean the format passed
        return "give_feedback"

    return route_after_format


# ---------- Connect the components ----------

def build_helper(model=None, max_retries: int = 3, feedback_char_limit: int = 450):
    """With model=None it uses fixed text. max_retries is a legacy parameter name; it actually counts generations, including the first."""
    give_feedback = make_give_feedback(model, feedback_char_limit) if model else give_feedback_fixed_text

    builder = StateGraph(State)
    builder.add_node("check_syntax", check_syntax)
    builder.add_node("explain_error", explain_error)
    builder.add_node("give_feedback", give_feedback)
    builder.add_node("check_format", make_check_format(feedback_char_limit))

    builder.add_edge(START, "check_syntax")
    builder.add_conditional_edges("check_syntax", route_after_syntax, ["give_feedback", "explain_error"])
    builder.add_edge("explain_error", END)
    builder.add_edge("give_feedback", "check_format")
    builder.add_conditional_edges("check_format", make_route_after_format(max_retries), ["give_feedback", END])
    return builder.compile()


def blank_order_slip(code: str) -> State:
    return {"code": code, "syntax_ok": None, "error": "", "feedback": "", "format_ok": None,
            "format_problem": "", "retries": 0}
''',
    'skill_loader.py': r'''"""skill_loader: the AI picks a handbook by looking only at its "card", and the body is opened only after it is picked.

Progressive disclosure has three layers:
1. Card: the name and description between the two --- lines at the top of SKILL.md. They are given to the model when it picks
2. Body: the whole SKILL.md, read only after it is picked
3. Appendix: files in scripts/ and references/, opened only when needed
"""

from pathlib import Path

SKILLS = Path(__file__).resolve().parent / "skills"


def read_card(path: Path) -> dict:
    """Read only the name and description between the two --- lines."""
    text = Path(path).read_text(encoding="utf-8")
    card = {"name": Path(path).parent.name, "description": "", "path": str(path)}
    if text.startswith("---"):
        head = text.split("---", 2)[1]
        for line in head.strip().splitlines():
            key, _, value = line.partition(":")
            if key.strip() in ("name", "description"):
                card[key.strip()] = value.strip()
    return card


def read_body(card: dict) -> str:
    """Opened only after the handbook is picked: the handbook content below the card."""
    text = Path(card["path"]).read_text(encoding="utf-8")
    return text.split("---", 2)[2].strip() if text.startswith("---") else text


def list_cards(only: list[str] | None = None) -> list[dict]:
    cards = [read_card(p) for p in sorted(SKILLS.glob("*/SKILL.md"))]
    if only:
        cards = [c for c in cards if Path(c["path"]).parent.name in only]
    return cards


def pick_handbook(model, request: str, cards: list[dict]) -> dict:
    """Show the model all the cards and let it pick one. Returns the card that was picked."""
    if not cards:
        raise ValueError("No handbook is available. Write a SKILL.md first, then list the cards.")
    card_list = "\n".join(f"- {c['name']}: {c['description']}" for c in cards)
    answer = model.invoke(
        "Pick the one handbook that best fits the review request below.\n"
        f"Review request: {request}\n\n"
        f"Handbook cards:\n{card_list}\n\n"
        "Answer with only the handbook's name, no other words."
    ).text.strip()
    name = answer.strip("`\"' \n")
    options = {c["name"]: c for c in cards}
    if name not in options:
        raise ValueError(f"The model did not return exactly one valid handbook name: {answer!r}. Valid names: {list(options)}. Check the cards and the model output; no handbook was picked this time.")
    return options[name]
''',
    'skills/friendly-feedback/SKILL.md': r'''---
name: friendly-feedback
description: Gentle, encouraging homework feedback for Python beginners. Starts with strengths, then suggests improvements. Good for first-time submissions and students who just learned Python.
---

# Friendly Feedback Rules

1. Start with one thing the student did well
2. Point out at most two problems, and give a fix for each
3. Don't use words like "wrong" or "terrible"
4. Write the feedback in two paragraphs: the first starts with "Strengths:", the second starts with "Improvements:"
''',
    'skills/strict-review/SKILL.md': r'''---
name: strict-review
description: Strict, thorough review for midterms and formal grading. Lists every problem, no softening, and includes point deductions.
---

# Strict Review Rules

1. List every problem, miss none, and give the line number for each
2. No pleasantries, and no words like "great" or "nice"
3. Give the points deducted for each problem (100 points in total)
4. Write the feedback in two paragraphs: the first starts with "Strengths:" (if there really are none, write "Strengths: none"), the second starts with "Improvements:"
''',
    'skills/naming-check/SKILL.md': r'''---
name: naming-check
description: Checks whether the variable and function names in a Python homework are lowercase English with underscores (snake_case). Use it when reviewing a homework and you need to list the naming problems.
---

# Naming Check

Run scripts/check_naming.py (the tool is named check_naming) and write its output into the feedback.
Don't judge by yourself, and don't add or drop any item. Copy every line it prints.

Shared rules: check assigned variables, function names and function parameters; for repeated names, list only the first definition; don't check class names; skip self and cls.
A name may have one leading underscore, followed by a lowercase English letter, and then lowercase English letters, digits or underscores.
This exercise has no exception for constants: MAX fails; result_2 passes. The script runs these rules.

Output format: copy every line the script prints, for example "Line 3: myVar breaks the rule (suggest my_var)"
''',
    'skills/naming-check/scripts/check_naming.py': r'''"""Check whether variable and function names are lowercase English with underscores (snake_case).

Usage:
    python check_naming.py homework.py        # check one file
    python check_naming.py < homework.py      # or read from standard input
Output:
    One line per problem, for example "Line 3: myVar breaks the rule (suggest my_var)"
    When everything passes it prints "OK: all variable and function names are lowercase with underscores"

A script does one thing and prints only a few short lines, so people and the AI can both check it. The same input and the same script give the same result.
"""

import ast
import re
import sys

RULE = re.compile(r"^_?[a-z][a-z0-9_]*$")


def suggest_name(name: str) -> str:
    s = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)
    return re.sub(r"_+", "_", s).lower()


def find_all_names(tree: ast.AST) -> dict[str, int]:
    """name → line number of its first appearance (variables, functions, parameters; class names are not included)."""
    names = {}

    def note(name, line):
        if name not in names or line < names[name]:
            names[name] = line

    for node in ast.walk(tree):
        if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store):
            note(node.id, node.lineno)
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            note(node.name, node.lineno)
        elif isinstance(node, ast.arg) and node.arg not in ("self", "cls"):
            note(node.arg, node.lineno)
    return names


def check_names(code: str) -> str:
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"Cannot check: line {e.lineno} has a syntax error"
    problems = [
        f"Line {line}: {name} breaks the rule (suggest {suggest_name(name)})"
        for name, line in sorted(find_all_names(tree).items(), key=lambda x: (x[1], x[0]))
        if not RULE.match(name)
    ]
    return "\n".join(problems) if problems else "OK: all variable and function names are lowercase with underscores"


if __name__ == "__main__":
    if len(sys.argv) > 1:
        with open(sys.argv[1], encoding="utf-8") as f:
            source = f.read()
    else:
        source = sys.stdin.buffer.read().decode("utf-8")
    sys.stdout.reconfigure(encoding="utf-8")
    print(check_names(source))
''',
}
for rel, content in _files.items():
    (RB / rel).write_text(content, encoding="utf-8")
sys.path.insert(0, str(RB))
# Make Python look again for the files we just wrote (if another cell ran first, Python remembers "this folder does not exist")
import importlib
importlib.invalidate_caches()

# 3. Remember the model source (the API key is not saved) and test the connection
import llm
llm.save_settings(model_source, colab_model_name.strip(), gemini_model_name.strip())
if INSTRUCTOR_API_KEY.strip():
    os.environ["GOOGLE_API_KEY"] = INSTRUCTOR_API_KEY.strip()
from workshop import *
model = prepare()

reply = model.invoke("Say hello in one sentence to a classmate who just learned Python").text
print(f"✅ Setup complete. Model source: {model_source}")
print(f"🤖 The model says: {reply.strip()}")

---
## Part 1: How the workflow and the agent fit together

> 📍 **Matching slides**: slide 25 "Combining workflow and agent: wiring handbooks and tools", slide 26 "Where workflow meets agent: load the handbook, provide the tools"

First look at the combined flow chart. The next cell writes `dual_core.py` (no changes needed; just run it).
The file name `dual_core` is an old name. It means slide 25's "combining workflow and agent", the "dual-core" in the course title. If you want to read the code, look at three places:

- `build_integrated_helper`: the outer nodes (the stations) and arrows, almost the same as the 1-A workflow, with one extra station, `skill_loader`
- `make_skill_loader`: shows the model the **cards** of all handbooks, picks one, and puts its body into the State's `skill`
- `make_give_feedback`: inside it, `create_agent` (the one-line way to make an agent) builds a small agent,
  with the handbook as its system prompt (the text that always goes first and tells the AI what to follow), and a toolbox of `check_naming`, `run_code` and `read_spec`

### 🤔 Prediction questions
1. In the outer flow chart, between which two stations will `skill_loader` sit?
2. Can you see in the flow chart which tools the inner agent will call?

In [ ]:
%%writefile /content/review-bot/dual_core.py
"""2-D: the integrated Homework Helper. The outside is the 1-A workflow (branches and order controlled by code), and the inside is the 1-B agent (decided by the AI).

START → check_syntax ─syntax OK→ skill_loader → give_feedback → check_format ─OK→ END
                     └syntax error→ explain_error → END            └failed and limit not reached→ send back to give_feedback

- skill_loader : looks at the cards of all handbooks, picks one, and puts its body into the State (the handbooks from 2-A)
- give_feedback: contains a small agent that uses the picked handbook as its system prompt,
                 and its toolbox has run_code, read_spec, check_naming (the 2-B script wrapped as a tool)
- checkpointer : saves the State of every step into a SQLite file, so the same thread_id can continue running
"""

import operator
import sqlite3
from pathlib import Path
from typing import Annotated, TypedDict

from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph

from graph import check_syntax, explain_error, route_after_syntax, make_check_format, make_route_after_format
from skill_loader import list_cards, pick_handbook, read_body
from tools.check_naming import check_naming
from tools.run_code import run_code

CHECKPOINT_PATH = Path(__file__).resolve().parent / "checkpoints.sqlite"
AVAILABLE_HANDBOOKS = ["friendly-feedback", "strict-review"]
sample_specs = {
    "Correct program": "Write a function add(a, b) that returns the sum of two numbers, and use print(add(2, 3)) to print 5.",
    "Variable typo": "Write a function add(a, b) that returns the sum of two numbers, and use print(add(2, 3)) to print 5.",
    "Missing parenthesis": "Use print to print hi.",
    "Inconsistent naming": "Write a function that computes the average of a list of scores. Test it with [90, 85, 77] and print the average along with the full mark 100 for comparison. Name variables, functions and parameters in lowercase with underscores.",
    "Messy indentation": "Write an average(scores) function that computes the average of the scores, and print the result of average([90, 85, 77]).",
}


class IntegratedState(TypedDict, total=False):
    code: str                 # the homework the student submitted
    spec: str                 # the assignment paired with this homework, saved along with the State
    request: str              # the review request (skill_loader uses it to pick a handbook)
    syntax_ok: bool | None
    error: str
    skill_name: str           # the handbook that was picked
    skill: str                # the handbook body
    feedback: str
    format_ok: bool | None
    format_problem: str
    retries: int
    tools_used: Annotated[list[str], operator.add]  # which tools the inner agent used (keeps appending)


def homework_input(homework_name: str, code: str, review_request: str) -> IntegratedState:
    """Pair the assignment when a new run starts. A resumed run uses the checkpoint and does not re-read the form or shared files."""
    if homework_name not in sample_specs:
        raise ValueError(f"No assignment is set up for this homework yet: {homework_name}")
    return {"code": code, "spec": sample_specs[homework_name], "request": review_request, "retries": 0}


def make_read_spec(spec_text: str):
    """The agent still chooses whether to use the tool, but what it reads is fixed to the assignment in that State."""
    if not isinstance(spec_text, str) or not spec_text.strip():
        raise ValueError("The State has no assignment (spec). Use homework_input to start a new run. For an old checkpoint, use a brand-new thread_id and don't fall back to the shared spec.md.")

    @tool
    def read_spec() -> str:
        """Read the assignment requirements paired with this homework. Use it before judging whether the homework meets the assignment. It takes no parameters."""
        return spec_text

    return read_spec


def make_skill_loader(model):
    def skill_loader(state: IntegratedState) -> dict:
        card = pick_handbook(model, state.get("request", ""), list_cards(AVAILABLE_HANDBOOKS))
        return {"skill_name": card["name"], "skill": read_body(card)}

    return skill_loader


def make_give_feedback(model, feedback_char_limit: int):
    def give_feedback(state: IntegratedState) -> dict:
        read_spec = make_read_spec(state.get("spec", ""))
        system_prompt = (
            f"{state['skill']}\n\n"
            "# Your tools\n"
            "- check_naming: checks variable naming (a script that comes with the handbook). Use it when naming comes up, and copy its result as is\n"
            "- run_code: runs the homework and shows the output\n"
            "- read_spec: reads the assignment requirements\n"
            "You decide whether to use a tool and which one.\n\n"
            "# Format (the outer workflow checks it with code and sends back anything that fails)\n"
            '- Write only two paragraphs: the first starts with "Strengths:", the second starts with "Improvements:"\n'
            f'- The whole feedback (including "Strengths:", "Improvements:", and line breaks) must be at most {feedback_char_limit} characters'
        )
        if state.get("format_problem"):
            system_prompt += f"\n\nThe previous version was rejected because: {state['format_problem']}. Please fix it."
        inner_agent = create_agent(model, tools=[check_naming, run_code, read_spec], system_prompt=system_prompt)
        result = inner_agent.invoke(
            {"messages": [{"role": "user", "content": f"Please review this homework:\n```python\n{state['code']}\n```"}]},
            config={"recursion_limit": 25},
        )
        messages = result["messages"]
        used = [c["name"] for m in messages for c in (getattr(m, "tool_calls", None) or [])]
        return {"feedback": messages[-1].text.strip(), "retries": state.get("retries", 0) + 1, "tools_used": used}

    return give_feedback


def run_station_by_station(helper, inputs, config: dict) -> dict:
    """Run one station at a time, print which fields each station wrote, and return the State in the checkpoint when it finishes (or is interrupted)."""
    for event in helper.stream(inputs, config=config, stream_mode="updates"):
        for station, writes in event.items():
            if station == "__interrupt__":
                print("⏸️  Interrupted: the flow stops here, and the State is already saved in the checkpoint")
                continue
            print(f"▶ {station}")
            for key, value in (writes or {}).items():
                text = value if isinstance(value, str) else repr(value)
                if key == "skill":
                    text = text.splitlines()[0] + " ... (handbook body)"
                print(f"    {key}: {text[:200]}")
    return helper.get_state(config).values


def open_checkpointer(path: Path = CHECKPOINT_PATH) -> SqliteSaver:
    """Save the State into a SQLite file. As long as the same file still exists, you can read it back by thread_id."""
    return SqliteSaver(sqlite3.connect(str(path), check_same_thread=False))


def build_integrated_helper(model, checkpointer=None, max_retries: int = 3, feedback_char_limit: int = 750,
                interrupt_before: list[str] | None = None):
    builder = StateGraph(IntegratedState)
    builder.add_node("check_syntax", check_syntax)
    builder.add_node("explain_error", explain_error)
    builder.add_node("skill_loader", make_skill_loader(model))
    builder.add_node("give_feedback", make_give_feedback(model, feedback_char_limit))
    builder.add_node("check_format", make_check_format(feedback_char_limit))

    builder.add_edge(START, "check_syntax")
    # If the syntax is OK, go pick a handbook first, then give feedback
    builder.add_conditional_edges(
        "check_syntax", route_after_syntax, {"give_feedback": "skill_loader", "explain_error": "explain_error"}
    )
    builder.add_edge("explain_error", END)
    builder.add_edge("skill_loader", "give_feedback")
    builder.add_edge("give_feedback", "check_format")
    builder.add_conditional_edges("check_format", make_route_after_format(max_retries), ["give_feedback", END])
    return builder.compile(checkpointer=checkpointer, interrupt_before=interrupt_before)


# Compatibility aliases for calls from an older version of the code; the new materials use the "integrated" names.
DualCoreState = IntegratedState
build_dual_core_helper = build_integrated_helper


In [ ]:
#@title ▶ Run: draw the outer flow chart of the combined helper
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import dual_core

helper = dual_core.build_integrated_helper(model)
show_graph(helper)

### 👀 What did you see?

- `skill_loader` sits between `check_syntax` (syntax OK) and `give_feedback`: only after the syntax passes does it pick a handbook, give feedback and check the format; a syntax error is explained and the run ends. **The workflow controls the order of this branch.**
- The flow chart does not show which tools the agent will call, because that happens **inside** the `give_feedback` station, and the AI decides it at that moment.
  The outside guarantees the required checks run; the inside keeps room for judgment. That is dual-core.

---
## Part 2: **Demo**: run the whole thing once

> 📍 **Matching slides**: slide 25 "Combining workflow and agent: wiring handbooks and tools"

The last part only drew the chart. In this part the instructor demonstrates a real run of the whole flow. The output prints station by station: the name after `▶` is the station, and the indented lines below it are the fields that station wrote into the State.

There are three kinds of outcome in this course. Please keep "ended" and "succeeded" apart:

| Path | Output and state |
| --- | --- |
| Syntax error | The code produces "Syntax: / Improvements:", does not enter the agent, and does not run the format check |
| Syntax passes, feedback format passes | The feedback uses "Strengths: / Improvements:", format_ok=True |
| Still not passed at the generation limit | The last version is kept for inspection, format_ok=False; it is not treated as a success |

### 🤔 Prediction questions
1. With the homework "Inconsistent naming" and the request for a beginner, which handbook will `skill_loader` pick? What about the midterm request?
2. Will the inner agent use `check_naming`? Will it also use `run_code`? If it uses `check_naming`, will the feedback include all 6 items the script found?
3. If the feedback it writes is too long, who sends it back?

In [ ]:
#@title ▶ Run: the integrated helper, station by station
homework = "Inconsistent naming"  #@param ["Inconsistent naming", "Variable typo", "Correct program", "Missing parenthesis"]
review_request = "Take a look for a beginner handing in homework for the first time"  #@param ["Take a look for a beginner handing in homework for the first time", "Midterm exam, official grading: please review strictly"] {allow-input: true}
feedback_char_limit = 750  #@param {"type":"integer"}
#@markdown **Maximum generations (including the first)**
max_generations = 3  #@param {"type":"integer"}
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import time
import dual_core

helper = dual_core.build_integrated_helper(model, checkpointer=dual_core.open_checkpointer(),
                                           max_retries=max_generations, feedback_char_limit=feedback_char_limit)
config = {"configurable": {"thread_id": f"full-demo-{time.time_ns()}"}}
state = dual_core.run_station_by_station(helper, dual_core.homework_input(homework, load_homework(homework), review_request), config)

print("\n========== Result ==========")
print("This homework:", homework)
print("This assignment:", state.get("spec"))
print("Handbook picked:", state.get("skill_name"))
print("Tools the inner agent used:", state.get("tools_used") or "(no tools used)")
print(f"Feedback generated {state.get('retries', 0)} time(s); format_ok = {state.get('format_ok')}")
print("Why it stopped:", "syntax did not pass" if not state.get("syntax_ok") else
      "format passed" if state.get("format_ok") else "reached the generation limit, format still not passed")
print("\nFeedback:")
print(state.get("feedback"))
record(f"2-D Part 2: one full run ({homework} / {review_request})", {k: v for k, v in state.items() if k != "skill"})

### 👀 What did you see?

- Syntax passes: `check_syntax → skill_loader → give_feedback → check_format`; it goes back to `give_feedback` only if the format fails and the generation limit has not been reached.
- Syntax fails: `check_syntax → explain_error → end`. No handbook is picked, the model is not called, and `check_format` is skipped. To watch this branch, change only the homework to "Missing parenthesis"; afterward set it back to "Inconsistent naming".
- Handbook choice, tool use and the feedback content are still up to the model. The required checks ran, but that does not mean the result passes.
- This cell prints only which tools were used, not what they returned. For "Inconsistent naming" the script lists 6 items (you saw them in 2-B Part 1). Compare that with the feedback: it may list all of them or pick only a few, and the character limit also affects how many it picks. **Using a tool does not mean the feedback relays the tool's result in full.**
- Optional limit comparison: set the homework back to "Inconsistent naming", keep the request for a beginner, set maximum generations to **1** and the character limit to **750**, and run once as the baseline. In the second round **change only the character limit to 1**. The two-part format can't fit in one character, so you should see "reached the limit, format still not passed". Afterward restore **750 / 3**.
- `retries` is an internal field name. In this implementation it goes up by one every time feedback is written, **including the first draft**; in class, read it as "generations so far".

---
## Part 3: **Experiment**: interrupt, stop halfway

> 📍 **Matching slides**: slide 27 "Pause once, then resume with the same ID"

Part 2 ran all the way through. But in the real world connections drop and programs crash. In this part you run it yourself and stop it halfway on purpose, to see whether the State was saved.

A **checkpointer** saves the whole order slip (the State) each time a station finishes; a **thread_id** is the number of that save, like a locker number.
The saves live in the file `/content/review-bot/checkpoints.sqlite`. As long as that file exists, re-running a cell can read them back. Colab may delete or replace the runtime and remove local files; if the file is gone, there is no old checkpoint.

This part fixes the homework to "Inconsistent naming" (valid syntax) with the beginner request, and saves the matching average assignment. With a brand-new thread_id, the cell below pauses **before** `give_feedback`. This is a planned teaching interruption, not a simulation of every way a real disconnect can be recovered; a syntax error takes a different branch and cannot stop here.

### 🤔 Prediction question
When it stops, which fields of the State in the checkpoint are already filled in? Does `feedback` have content?

In [ ]:
#@title ▶ Run: stop just before give_feedback
thread_id = "Alex's homework"  #@param {type:"string"}
homework = "Inconsistent naming"  #@param ["Inconsistent naming", "Variable typo", "Correct program", "Missing parenthesis"]
review_request = "Take a look for a beginner handing in homework for the first time"  #@param ["Take a look for a beginner handing in homework for the first time", "Midterm exam, official grading: please review strictly"] {allow-input: true}
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import dual_core

helper = dual_core.build_integrated_helper(model, checkpointer=dual_core.open_checkpointer(), interrupt_before=["give_feedback"])
config = {"configurable": {"thread_id": thread_id}}
old_checkpointer = helper.get_state(config)
if old_checkpointer.values and not old_checkpointer.values.get("spec"):
    raise SystemExit("This is an old checkpoint that did not save the assignment. In Parts 3 and 4, switch to the same brand-new thread_id, then pause and resume; the original checkpoint is kept.")
if old_checkpointer.values:
    print(f"This ID already has a checkpoint: {'not finished' if old_checkpointer.next else 'finished'}. This cell keeps the original content and does not submit the homework again.")
    print("To resume, go to Part 4; to reset the experiment, fill the same brand-new ID in Parts 3 and 4, for example \"Alex's homework-2\".")
else:
    dual_core.run_station_by_station(helper, dual_core.homework_input(homework, load_homework(homework), review_request), config)
checkpointer = helper.get_state(config)
print(f"\n📂 Checkpoint for thread_id \"{thread_id}\":")
print_state({k: v for k, v in checkpointer.values.items() if k not in ("skill", "code")})
print("Next station (not run yet):", checkpointer.next)
record(f"2-D Part 3: interrupt (thread_id={thread_id})", {"next": list(checkpointer.next), **{k: v for k, v in checkpointer.values.items() if k != "skill"}})

### 👀 What did you see?

- With a brand-new ID and the default homework, it stops before `give_feedback`: `syntax_ok` and `skill_name` are filled in, and `feedback` is still empty.
- The assignment is also saved in `spec`: this average assignment will not pick up the addition assignment left over from the previous notebook. A resumed run uses the checkpoint, not the new assignment in the current form. For an old checkpoint with no `spec`, use a brand-new ID; the original is not deleted.
- The "next station" is `('give_feedback',)`: the checkpoint remembers not only the State but also **where the flow had got to**.

---
## Part 4 ⭐: **Experiment**: resume, or start from blank?

> 📍 **Matching slides**: slides 25-28; minimum to complete: tell apart resuming an unfinished thread, retrieving a finished result, and a fresh run on a new ID

The last part stopped halfway and left an "unfinished" State in the checkpoint. In this part you use the thread_id to look for a checkpoint:
if it finds one stopped halfway, it **resumes**; if it is already finished, it **only reads the existing result**; only a brand-new ID **starts from blank**.
Slide 25 has a table that compares these three cases.

| ID status | What it does | Re-reads this cell's homework fields? | Slide |
| --- | --- | --- | --- |
| Old ID, not finished | Continues with the pending node | No, uses the homework in the checkpoint | 27 |
| Old ID, finished | Shows the finished result | No, does not review again | 28 |
| A new ID never used | Starts from check_syntax | Yes | 25 |

### 🤔 Prediction questions (three rounds; write the answers on paper)

Follow the slides in two steps: do the first round on slide 27; once resuming works, go to slide 28 for the second and third rounds.
1. Keep the thread_id as **Alex's homework** (same as in Part 3) and run. Which station does it start from? Does `skill_name` need to be picked again?
2. Run again with the same ID: will the finished result be reviewed again?
3. Change the thread_id to a **never-used** "Ben's homework" and run again. Which station does it start from this time?

In [ ]:
#@title ▶ Run: look up the checkpoint by thread_id, then resume or start from blank
thread_id = "Alex's homework"  #@param {type:"string"}
homework = "Inconsistent naming"  #@param ["Inconsistent naming", "Variable typo", "Correct program", "Missing parenthesis"]
review_request = "Take a look for a beginner handing in homework for the first time"  #@param ["Take a look for a beginner handing in homework for the first time", "Midterm exam, official grading: please review strictly"] {allow-input: true}
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import dual_core

helper = dual_core.build_integrated_helper(model, checkpointer=dual_core.open_checkpointer())
config = {"configurable": {"thread_id": thread_id}}
checkpointer = helper.get_state(config)

if checkpointer.values and not checkpointer.values.get("spec"):
    raise SystemExit("This is an old checkpoint that did not save the assignment. Go back to Part 3, build a paused state with a brand-new thread_id, then fill in the same ID to resume.")
if not checkpointer.values:
    print(f"🆕 thread_id \"{thread_id}\" has no checkpoint: the State is blank, so the run starts from the beginning\n")
    state = dual_core.run_station_by_station(helper, dual_core.homework_input(homework, load_homework(homework), review_request), config)
    situation = "no checkpoint, started from blank"
elif checkpointer.next:
    print(f"📂 Found a checkpoint for thread_id \"{thread_id}\": stopped before {checkpointer.next}, resuming")
    print("   Already in the checkpoint:", {k: v for k, v in checkpointer.values.items() if k in ("syntax_ok", "skill_name", "retries")}, "\n")
    state = dual_core.run_station_by_station(helper, None, config)      # input None = continue from the checkpoint
    situation = "found a checkpoint, resumed"
else:
    print(f"📂 thread_id \"{thread_id}\" has already finished; the checkpoint content is below (to start over, use a different thread_id)\n")
    state = checkpointer.values
    situation = "already finished"

print("\n========== Current State ==========")
print("Assignment in the checkpoint:", state.get("spec"))
print_state({k: v for k, v in state.items() if k not in ("skill", "code")})
print("Next station in the latest checkpoint:", helper.get_state(config).next or "(the flow has ended)")
print("Reading the result:", "syntax did not pass; the feedback format was not checked" if state.get("syntax_ok") is False else
      "feedback format passed; the content still needs a look" if state.get("format_ok") else "feedback format did not pass; the flow ending does not mean success")
record(f"2-D Part 4: {situation} (thread_id={thread_id})", {k: v for k, v in state.items() if k != "skill"})

### 👀 What did you see?

- **The same unfinished thread_id**: it continues from `give_feedback`; `check_syntax` and `skill_loader` are not re-run, and `skill_name` is the one that was in the checkpoint. **The State was not lost.**
- **A thread_id never used**: the checkpoint is empty, the State starts from blank, and the whole flow runs again from `check_syntax`.
- Here "finished" only means the flow has no next station; it does not mean the homework or the feedback passes. If you run the same thread_id again after it finishes, you will see "already finished": this time it only shows the original result. Switching IDs just looks up a different checkpoint; it does not delete the original one.
- **Completion check**: point out the checkpoint ID, the next station and the skill_name in the output, and say whether the three rounds picked a handbook again. To reset from blank, change both cells to a brand-new ID.

---
## Part 5: Inspect the run history: the State history of one thread

> 📍 **Matching slides**: slide 28 (look at the State history after the run finishes)

Part 4 looked at the "latest" checkpoint. In fact the checkpointer **saves one checkpoint every time a station finishes**, and the cell below lists every checkpoint of the same thread_id.
This is **viewing the State history**; it does not re-run any node. Replay (re-running from a specific checkpoint) is an advanced operation that this notebook does not demonstrate.

### 🤔 Prediction question
Find which checkpoint first has skill_name and which first has feedback. The number of rewrites affects the number of checkpoints.

In [ ]:
#@title ▶ Run: list every checkpoint of one thread_id
thread_id = "Alex's homework"  #@param {type:"string"}
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import dual_core

helper = dual_core.build_integrated_helper(model, checkpointer=dual_core.open_checkpointer())
config = {"configurable": {"thread_id": thread_id}}
history = list(helper.get_state_history(config))
if not history:
    print(f"thread_id \"{thread_id}\" has no checkpoint. Run Parts 3 and 4 first.")
for snap in reversed(history):
    v = snap.values
    step_no = snap.metadata.get("step")
    print(f"{'input received' if step_no == -1 else f'step {step_no}'} | next station: {snap.next or '(end)'} | "
          f"syntax_ok={v.get('syntax_ok')}  skill={v.get('skill_name')}  "
          f"retries={v.get('retries')}  format_ok={v.get('format_ok')}  has feedback={'yes' if v.get('feedback') else 'no'}")
record(f"2-D Part 5: State history (thread_id={thread_id})",
      [{"step": s.metadata.get("step"), "next": list(s.next),
        **{k: val for k, val in s.values.items() if k not in ("skill", "code")}} for s in reversed(history)])

### 👀 What did you see?

- Every time a station finishes, one more checkpoint appears, so you can watch the order slip fill in field by field. It is much like 1-A printing the whole order slip step by step, except this time it is **stored in a file**.
- This cell is **viewing history**; it does not re-run any node.

---
## Part 6 (optional): Eval: compare using a separate pure agent

> 📍 **Matching slides**: optional slide 43 "Optional: checking a pure agent with code", 44 "Optional: a fixed test set, before vs. after a change"

This part **uses a different, pure agent for testing**, not the integrated helper from Parts 1-5.
A pure agent has no outer flow control; it decides on its own whether to check syntax first and which tools to call. How do you check how well it does? **Test it.**

Writing "what the agent should do" as a test that code can grade automatically is called an **eval**. Here are five test cases (eval cases),
each tied to its own assignment requirements, so the average assignment never reads the addition spec. Each case checks two things: **which tool is called first** and **which words the final reply must contain**. Code grades the test, not a person. We compare two versions of the system prompt:

- **baseline** (the original version, used as the reference point): says nothing special about order
- **v2** (the improved version): adds one sentence, "always call check_syntax before giving feedback" (here `check_syntax` is the helper's syntax check wrapped as a tool)

Note: this "eval" tests an agent with test cases. It is different from the "verifying agent" on Session 2 slide 35 (which checks whether each piece of feedback makes things up). The form field "run_verification" means whether to run this eval.

⚠️ This part calls the model about 20-30 times. If your free quota is small, **run it after class**: tick "run_verification" in the form and run the cell.

### 🤔 Prediction questions
How many of the five will baseline pass? What about after adding the sentence? Will it hit exactly 100%?

In [ ]:
#@title ▶ Run (optional): five test cases, compare the pass rate of baseline and v2
run_verification = False  #@param {type:"boolean"}
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
from langchain.agents import create_agent
from langchain.tools import tool
import graph
import dual_core
from tools.run_code import run_code

@tool
def check_syntax(code: str) -> str:
    """Check whether a Python homework's syntax is correct. Returns "Syntax OK" or the line where the error is."""
    result = graph.check_syntax({"code": code})
    return "Syntax OK" if result["syntax_ok"] else f"Syntax error: {result['error']}"

# Five test cases: who must be called first, and what the final reply must contain
EVAL_CASES = [
    {"id": "001_must_check_syntax_first", "homework": "Correct program", "must_call_first": "check_syntax", "must_contain_in_final": ["Strengths", "Improvements"]},
    {"id": "002_handle_syntax_error", "homework": "Missing parenthesis", "must_call_first": "check_syntax", "must_contain_in_final": ["Syntax"]},
    {"id": "003_wrong_variable", "homework": "Variable typo", "must_call_first": "check_syntax", "must_contain_in_final": ["Improvements"]},
    {"id": "004_messy_indent", "homework": "Messy indentation", "must_call_first": "check_syntax", "must_contain_in_final": ["Improvements"]},
    {"id": "005_naming", "homework": "Inconsistent naming", "must_call_first": "check_syntax", "must_contain_in_final": ["Strengths", "Improvements"]},
]

def evaluate_one(system_prompt, case):
    read_spec = dual_core.make_read_spec(dual_core.sample_specs[case["homework"]])
    candidate = create_agent(model, tools=[check_syntax, run_code, read_spec], system_prompt=system_prompt)
    inputs = {"messages": [{"role": "user", "content": f"Please review this homework:\n```python\n{load_homework(case['homework'])}\n```"}]}
    events = list(candidate.stream(inputs, stream_mode="updates"))
    msgs = [m for e in events for u in e.values() for m in (u or {}).get("messages", [])]
    tc = [c["name"] for m in msgs for c in (getattr(m, "tool_calls", None) or [])]   # order of tool calls
    final = next((m.text for m in reversed(msgs) if m.type == "ai" and not m.tool_calls), "")
    reasons = []
    if not tc:
        reasons.append(f"called no tool (expected {case['must_call_first']})")
    elif tc[0] != case["must_call_first"]:
        reasons.append(f"first tool was {tc[0]}, expected {case['must_call_first']}")
    for kw in case["must_contain_in_final"]:
        if kw not in final:
            reasons.append(f"final reply is missing the keyword {kw!r}")
    return {"id": case["id"], "passed": not reasons, "reasons": reasons, "tool_calls": tc}

base = "You are a Python homework helper. Give beginners feedback in English. The feedback has two parts: the first starts with \"Strengths:\" and the second starts with \"Improvements:\"."
versions = {
    "baseline (no order given)": base,
    "v2 (adds \"call check_syntax before giving feedback\")": base + " Always call check_syntax to check the syntax before giving feedback.",
}

if not run_verification:
    print("This part is optional: tick \"run_verification\" in the form above, then run it again.")
else:
    grade = {}
    for version, system_prompt in versions.items():
        print(f"===== {version} =====")
        results = [evaluate_one(system_prompt, case) for case in EVAL_CASES]
        for r in results:
            print(f"[{'PASS' if r['passed'] else 'FAIL'}] {r['id']}  tool_calls={r['tool_calls']}")
            for why in r["reasons"]:
                print(f"        - {why}")
        passed = sum(r["passed"] for r in results)
        print(f"Passed {passed}/{len(results)} ({passed / len(results):.0%})\n")
        grade[version] = f"{passed}/{len(results)}"
    record("2-D Part 6: eval pass rate", grade)

### 👀 What did you see?

- First write down how many cases baseline and v2 each passed, and why the others failed. Baseline may pass all of them; record what you actually saw.
- v2 may improve, stay the same or get worse; one run of five cases only describes this test. Matching a keyword does not mean the feedback is correct, so you still need to spot-check the content by hand.
- This is an **eval of a pure agent**. Compare it with the **integrated** helper in Parts 1-5, which writes the "check order" into the outer flow and the "format rule" into the `check_format` node,
  so that code runs the required checks along its branches (a syntax error never enters feedback; the format is checked only after the syntax passes). The integrated helper still needs testing of its branches, its limit, checkpoint resuming and the quality of its feedback, which a plain eval cannot test.

## Wrap-up: save your work and hand it in

- 💾 **Save to Google Drive**: save a copy of `/content/review-bot/` (your edited handbooks, code, and State records) to your own Google Drive.
- ⬇️ **Download the zip**: download it straight to your computer.
- 📸 **The last cell**: prints the results the experiment cells saved. Pick the relevant records for a screenshot, then add your observations and reasons as the assignment asks.

These are the results the experiment cells saved through `record()`, not every intermediate state of every run.
Records in the same runtime pile up, so they may include experiments from other notebooks; use the printed titles to tell them apart.

A temporary browser disconnect does not necessarily wipe your files; files under `/content/` are lost only when the runtime is deleted or recycled.
To keep your handbooks, code, and checkpoint, back them up first. The notebook itself is saved with the menu "File → Save a copy in Drive". The two kinds of saving do different jobs.
When you come back, first check that the runtime is still there. If it was reset, re-run "0. Setup" and this part's setup cell, then restore the files you backed up as needed.

In [ ]:
#@title 💾 Save to Google Drive (an authorization window will pop up)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
save_to_drive()

In [ ]:
#@title ⬇️ Download the zip
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
download_zip()

In [ ]:
#@title 📸 Print the recorded experiment results (pick the relevant records to hand in)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
print_history()